In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [12]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load paths
rootDir = '/kaggle/input/indoor-scenes-cvpr-2019/indoorCVPR_09/Images'
allData = [(os.path.join(rootDir, label, file), label)
           for label in sorted(os.listdir(rootDir))
           for file in os.listdir(os.path.join(rootDir, label))
           if file.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Stratified split
trainData, tempData = train_test_split(allData, test_size=0.2, stratify=[lbl for _, lbl in allData], random_state=42)
valData, testData = train_test_split(tempData, test_size=0.5, stratify=[lbl for _, lbl in tempData], random_state=42)

# Labels
labelNames = sorted(set(lbl for _, lbl in allData))
labelToIndex = {lbl: idx for idx, lbl in enumerate(labelNames)}
classNames = labelNames


Using device: cuda


In [13]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

trainTransform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=15, shear=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

valTransform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class SceneDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        imgPath, label = self.data[idx]
        image = Image.open(imgPath).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, labelToIndex[label]


In [14]:
batchSize = 64
trainDataset = SceneDataset(trainData, transform=trainTransform)
valDataset   = SceneDataset(valData, transform=valTransform)
testDataset  = SceneDataset(testData, transform=valTransform)

trainLoader = DataLoader(trainDataset, batch_size=batchSize, shuffle=True)
valLoader   = DataLoader(valDataset, batch_size=batchSize)
testLoader  = DataLoader(testDataset, batch_size=batchSize)


Teacher Model

In [ ]:
import torch
import torch.nn as nn

# Dense Layer
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super(DenseLayer, self).__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv(self.relu(self.bn(x)))
        return torch.cat([x, out], 1)  

# Dense Block
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super(DenseBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            layers.append(DenseLayer(in_channels + i * growth_rate, growth_rate))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# Transition Layer
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(TransitionLayer, self).__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(2)
        )

    def forward(self, x):
        return self.layer(x)

#Full DenseNet
class DenseNetLikeSceneClassifier(nn.Module):
    def __init__(self, growth_rate=32, block_layers=[4, 6, 8], num_classes=67):
        super(DenseNetLikeSceneClassifier, self).__init__()

        # Initial Conv Layer
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),  # Conv
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)  # Pool
        )

        # Dense Blocks + Transitions
        self.features = nn.Sequential()
        num_channels = 64
        for i, num_layers in enumerate(block_layers):
            block = DenseBlock(num_layers, num_channels, growth_rate)
            self.features.add_module(f'denseblock{i+1}', block)
            num_channels = num_channels + num_layers * growth_rate
            if i != len(block_layers) - 1:  
                trans = TransitionLayer(num_channels, num_channels // 2)
                self.features.add_module(f'transition{i+1}', trans)
                num_channels = num_channels // 2

        # Final batch norm
        self.bn_final = nn.BatchNorm2d(num_channels)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(num_channels, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.bn_final(x)
        x = self.classifier(x)
        return x


In [6]:
teacherModel = DenseNetLikeSceneClassifier(growth_rate=32, block_layers=[6,12,24,16], num_classes=67)
teacherModel.load_state_dict(torch.load("/kaggle/input/bestmodel/pytorch/default/1/bestModel.pth", map_location=device))
teacherModel.to(device)
teacherModel.eval()

DenseNetLikeSceneClassifier(
  (stem): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (features): Sequential(
    (denseblock1): DenseBlock(
      (block): Sequential(
        (0): DenseLayer(
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (1): DenseLayer(
          (bn): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv): Conv2d(96, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (2): DenseLayer(
          (bn): Batch

Student Model

In [16]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        se = self.pool(x).view(b, c)
        se = self.fc(se).view(b, c, 1, 1)
        return x * se

class HybridEfficientCNN_MIT67(nn.Module):
    def __init__(self, num_classes=67):
        super(HybridEfficientCNN_MIT67, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            SEBlock(256),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

studentModel = HybridEfficientCNN_MIT67().to(device)


In [17]:
def distillation_loss(studentlogits, teacherlogits, labels, temperature=4.0, alpha=0.7):
    softTargetLoss = F.kl_div(
        F.log_softmax(studentlogits / temperature, dim=1),
        F.softmax(teacherlogits / temperature, dim=1),
        reduction='batchmean'
    ) * (temperature ** 2)

    hardTargetLoss = F.cross_entropy(studentlogits, labels)
    return alpha * softTargetLoss + (1 - alpha) * hardTargetLoss


In [18]:
def trainkd(studentmodel, teachermodel, trainloader, valloader, epochs=30, lr=1e-3):
    optimizer = torch.optim.AdamW(studentModel.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    bestAcc = 0
    criterionce = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        studentModel.train()
        totaltrainloss, traincorrect, traintotal = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            with torch.no_grad():
                teacher_logits = teacherModel(images)

            studentlogits = studentModel(images)
            loss = distillation_loss(studentlogits, teacher_logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            totaltrainloss += loss.item()
            preds = studentlogits.argmax(dim=1)
            traincorrect += (preds == labels).sum().item()
            traintotal += labels.size(0)

        trainacc = traincorrect / traintotal
        avgtrainloss = totaltrainloss / len(train_loader)

        # Validation
        studentModel.eval()
        totalvalloss, valcorrect, valtotal = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = studentModel(images)
                loss = criterionce(outputs, labels)
                totalvalloss += loss.item()
                preds = outputs.argmax(dim=1)
                valcorrect += (preds == labels).sum().item()
                valtotal += labels.size(0)

        valacc = valcorrect / valtotal
        avgvalloss = totalvalloss / len(val_loader)

        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Loss: {avgtrainloss:.4f} | Train Acc: {trainacc:.4f}")
        print(f"Val   Loss: {avgvalloss:.4f} | Val   Acc: {valacc:.4f}")

        if valacc > bestAcc:
            bestAcc = valacc
            torch.save(studentModel.state_dict(), "best_student_model.pth")
            print("Best model saved.")

        scheduler.step()

    print(f"\n Best Validation Accuracy: {bestAcc:.4f}")


In [20]:
# Dataset creation
train_dataset = SceneDataset(trainData, transform=trainTransform)
val_dataset   = SceneDataset(valData, transform=valTransform)
test_dataset  = SceneDataset(testData, transform=valTransform)

# DataLoader creation
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)


trainkd(studentModel, teacherModel, train_loader, val_loader, epochs=50)


Epoch 1/50: 100%|██████████| 391/391 [02:40<00:00,  2.44it/s]



[Epoch 1]
Train Loss: 2.3581 | Train Acc: 0.0969
Val   Loss: 3.6885 | Val   Acc: 0.1152
Best model saved.


Epoch 2/50: 100%|██████████| 391/391 [02:38<00:00,  2.46it/s]



[Epoch 2]
Train Loss: 2.2411 | Train Acc: 0.1184
Val   Loss: 3.4785 | Val   Acc: 0.1376
Best model saved.


Epoch 3/50: 100%|██████████| 391/391 [02:38<00:00,  2.46it/s]


...


[Epoch 48]
Train Loss: 1.1004 | Train Acc: 0.5703
Val   Loss: 1.6820 | Val   Acc: 0.5487


Epoch 49/50: 100%|██████████| 391/391 [02:38<00:00,  2.46it/s]



[Epoch 49]
Train Loss: 1.0958 | Train Acc: 0.5729
Val   Loss: 1.6807 | Val   Acc: 0.5487


Epoch 50/50: 100%|██████████| 391/391 [02:38<00:00,  2.46it/s]



[Epoch 50]
Train Loss: 1.0979 | Train Acc: 0.5713
Val   Loss: 1.6755 | Val   Acc: 0.5493

 Best Validation Accuracy: 0.5519


In [22]:
studentModel.load_state_dict(torch.load("best_student_model.pth"))
studentModel.eval()


HybridEfficientCNN_MIT67(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU

In [23]:
def evaluateModel(model, dataloader):
    model.eval()
    correct, total = 0, 0
    totalLoss = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            totalLoss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    avgLoss = totalLoss / len(dataloader)
    print(f"\nTest Accuracy : {accuracy * 100:.2f}% | Test Loss: {avgLoss:.4f}")
    return accuracy, avgLoss


In [24]:
evaluateModel(studentModel, test_loader)



Test Accuracy : 55.06% | Test Loss: 1.6853


(0.5505761843790012, 1.68527965642968)